# Multi‑Agent Systems & Coordination




In [2]:
# %%capture
# If needed in a fresh environment, uncomment and run:
!pip install -q langgraph langchain langchain_groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.1 MB/s eta 0:00:00


In [4]:
import getpass
import os

GROQ_API_KEY = getpass.getpass("Enter your Groq API Key: ")
os.environ["GROQ_API_KEY"] = GROQ_API_KEY
print("API Key set successfully!")

Enter your Groq API Key: ··········
API Key set successfully!


In [5]:
from typing import TypedDict, Literal, Dict, Any, List
from langchain_groq import ChatGroq
# from langchain.schema import SystemMessage, HumanMessage
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, START, END
from dataclasses import dataclass
import re


## Theory Primer: Single‑LLM Multi‑Role vs True Multi‑Agent Systems (MAS)

**Quick definitions**  
- **Single LLM, multi‑role:** one reasoning engine used sequentially with different *prompts/roles*. Usually one **shared session/state** unless you isolate it. Coordination is implicit (host code orders steps).  
- **MAS:** multiple **autonomous agents** (logical or physical) with **separate policies + private state** that **communicate via messages** and are **coordinated** (supervisor/graph/market). Shared artifacts (blackboard, DB) are OK; each agent still has **local memory/identity** and clear contracts.

| Axis | Single LLM, Multi‑Role | True Multi‑Agent |
|---|---|---|
| **Policy boundary** | One core policy, swapped by prompts | Distinct policies per agent |
| **Memory/state** | Often **shared** (same session/global) | **Isolated local memory** per agent; explicit shared resources |
| **Identity** | No stable addresses | Agent IDs / addressable mailboxes |
| **Coordination** | Linear orchestration in code | Explicit messaging + routing (supervisor/graph/queue) |
| **Autonomy** | Steps invoked by host code | Agents decide actions within constraints |
| **Failure isolation** | One failure contaminates all | Contained failures, retries, fallbacks |
| **Concurrency** | Usually sequential | Parallel branches, merges, consensus |
| **Observability** | One log/trace | Per‑agent logs/metrics/accountability |

**Rules of thumb**  
1. If agents have **IDs**, **private memory**, and you can **send(to=AgentID, …)** → it’s MAS.  
2. If you can **swap a role’s policy/model** without touching others → MAS.  
3. If all roles share the same hidden memory → likely single‑agent multi‑role.  
4. If a **supervisor/graph** decides next steps based on state → multi‑agent coordination.

**Patterns**  
- **Blackboard:** Shared artifacts (facts/docs) but **not** shared internal memory.  
- **Supervisor:** Router owns global process state; agents return artifacts.  
- **Peer/Swarm:** Agents negotiate with performatives (propose/confirm/reject).


###Example A — “Mailbox MAS” (FIPA-ish messages, private memory per agent)


* It defines four autonomous agents — Supervisor, Researcher, Analyst, Writer — each with its own Groq LLM instance and private memory.
* Agents don’t share hidden context; they communicate through explicit messages (request, inform, confirm, reject) that travel inside individual mailboxes.
* The Supervisor decides who acts next, enforcing a coordination policy: first ask the Researcher for findings → send them to Analyst for verification → forward cleaned results to Writer for summary.
* LangGraph handles the orchestration flow while each agent remains logically independent exactly like a miniature organization where members email each other under a manager’s plan.

In [7]:
# pip install langchain langchain_groq
from dataclasses import dataclass
from typing import Dict, List, Literal
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# ---- Agents: IDs, private LLM instances, private memories ----
AgentID = Literal["Supervisor","Researcher","Analyst","Writer"]
  #  Literal fixes a variable to a limited set of allowed values
ROLES = {
    "Researcher": "You are a rigorous market researcher. Output 3–5 numbered findings with short sources.",
    "Analyst":    "You are a strict analyst. Remove weak/uncited claims and justify each remaining in 1 line.",
    "Writer":     "You are a concise report writer. Produce a ~120-word executive summary, neutral tone.",
    "Supervisor":"You are a coordinator. You never write content; you only decide next step given message logs."
}

LLM: Dict[AgentID, ChatGroq] = {
    "Supervisor": ChatGroq(model="llama-3.1-8b-instant", temperature=0),
    "Researcher": ChatGroq(model="llama-3.1-8b-instant", temperature=0.2),
    "Analyst":    ChatGroq(model="llama-3.1-8b-instant", temperature=0),
    "Writer":     ChatGroq(model="llama-3.1-8b-instant", temperature=0.3),
}
#LLM is a dictionary whose keys are agent IDs and values are ChatGroq models
"""Every key must be one of the valid agents (‘Supervisor’, ‘Researcher’, etc.)
# and the value must be a ChatGroq object """

# private conversational memory per agent
MEM: Dict[AgentID, List] = {k: [] for k in LLM.keys()}
"""MEM is a dictionary where each key is an AgentID (like 'Researcher' or 'Writer'),
and the value is a List (that agent’s memory)"""




"MEM is a dictionary where each key is an AgentID (like 'Researcher' or 'Writer'),\nand the value is a List (that agent’s memory)"

In [8]:
def call(agent: AgentID, user: str) -> str:
    msgs = [SystemMessage(content=ROLES[agent])] + MEM[agent] + [HumanMessage(content=user)]
    out = LLM[agent].invoke(msgs).content
    MEM[agent].extend([HumanMessage(content=user), SystemMessage(content=f"[INTERNAL_STATE REDACTED]")])
    return out

    """This function makes an agent talk and remember. It takes which agent
    you want to use and what you want to ask.
    It adds that agent’s fixed role, its old messages, and our new question
    then sends all this to the agent’s LLM to get a reply.
    After getting the answer, it saves the new message into that agent’s private memory
    so next time it remembers what was said. Finally, it returns the agent’s reply text."""

In [9]:
#not memory, it is a message object —
#like an email that one agent sends to anothe
# ---- FIPA-ish message schema + mailboxes ----
@dataclass
class Msg:
    sender: AgentID
    receiver: AgentID
    performative: Literal["request","inform","propose","confirm","reject"]
    content: str
    conv_id: str


MAILBOX: Dict[AgentID, List[Msg]] = {k: [] for k in LLM.keys()}
# This creates an empty mailbox for every agent like giving each agent its own inbox to receive messages.


In [10]:
def send(m: Msg):
  MAILBOX[m.receiver].append(m)

#appends the message m into the receiver’s mailbox
# So after send() runs:
# MAILBOX = {
#   "Supervisor": [Msg(...from Researcher...)],
#   "Researcher": [],
#   "Analyst": [],
#   "Writer": []
# }



In [11]:
def recv(who: AgentID) -> Msg | None:
  return MAILBOX[who].pop(0) if MAILBOX[who] else None

  # Receive means check your inbox — if you have mail, read the first message; if not, return nothing

In [12]:
# ---- Kickoff (explicit message from Supervisor) ----
Q = "Pakistani K–12 EdTech market (2024–2025): size, growth, top 3 trends. Cite concise sources."
send(Msg("Supervisor","Researcher","request", Q, conv_id="MR-1001"))

* supervisor_policy() looks at the message log and checks what has already been done.
* If no message has come from the Researcher, it decides to start with the Researcher.
* If the Researcher has already sent findings but no Analyst message is found yet, it selects the Analyst next.
* If both Researcher and Analyst have sent their parts, it moves to the Writer.

In [13]:
# ---- Simple coordination policy (Supervisor decides next hop based on logs) ----
log: List[Msg] = []

def supervisor_policy(log: List[Msg]) -> AgentID | None:
    # If no Researcher -> request it; if findings exist but no Analyst review -> Analyst; if verified -> Writer.
    has_findings = any(m.sender=="Researcher" and m.performative in ("inform","confirm") for m in log)
    has_verified = any(m.sender=="Analyst" and m.performative in ("inform","confirm") for m in log)
    if not has_findings:  return "Researcher"
    if not has_verified:  return "Analyst"
    return "Writer"

log = [

  Msg("Supervisor", "Researcher", "request", "Find trends", "MR-1001"),

  Msg("Researcher", "Supervisor", "inform", "Here are 3 trends...", "MR-1001")

]




In [14]:
# ---- Agent loops (each agent reads its mailbox & responds) ----
def tick_researcher():
    msg = recv("Researcher")
    if not msg: return
    findings = call("Researcher", f"QUESTION: {msg.content}")
    out = Msg("Researcher","Supervisor","inform", findings, msg.conv_id); log.append(out); send(out)

def tick_analyst():
    msg = recv("Analyst")
    if not msg: return
    verified = call("Analyst", f"VERIFY & CLEAN:\n{msg.content}")
    out = Msg("Analyst","Supervisor","inform", verified, msg.conv_id); log.append(out); send(out)

def tick_writer():
    msg = recv("Writer")
    if not msg: return
    summary = call("Writer", f"SOURCE NOTES:\n{msg.content}")
    out = Msg("Writer","Supervisor","inform", summary, msg.conv_id); log.append(out); send(out)

def tick_supervisor():
    # Supervisor reviews latest log and routes next message explicitly to an agent
    nxt = supervisor_policy(log)
    if nxt == "Researcher":
        send(Msg("Supervisor","Researcher","request", Q, "MR-1001"))
    elif nxt == "Analyst":
        # pass only researcher findings (explicit artifact), not hidden memory
        last_findings = next(m for m in reversed(log) if m.sender=="Researcher").content
        send(Msg("Supervisor","Analyst","request", last_findings, "MR-1001"))
    elif nxt == "Writer":
        last_verified = next(m for m in reversed(log) if m.sender=="Analyst").content
        send(Msg("Supervisor","Writer","request", last_verified, "MR-1001"))
    else:
        return

# ---- Run a few cycles ----
for _ in range(6):
    tick_supervisor(); tick_researcher(); tick_analyst(); tick_writer()



In [15]:

# ---- Final artifact from Writer (explicit message) ----
final_msgs = [m for m in log if m.sender=="Writer"]
print("[FINAL SUMMARY]\n", final_msgs[-1].content if final_msgs else "(no output)")

[FINAL SUMMARY]
 **Executive Summary (120 words)**

The Pakistani K-12 EdTech market is expected to reach PKR 13.4 billion by 2025, growing at a CAGR of 20.5% from 2020 to 2025. The market growth is driven by increasing internet penetration, government initiatives, and the need for quality education. Key trends in the market include the adoption of digital learning platforms, which are expected to focus on online courses, virtual classrooms, and mobile learning apps. Additionally, the integration of Artificial Intelligence (AI) and Machine Learning (ML) in EdTech solutions is expected to enhance personalized learning experiences and improve student outcomes. These trends are justified by credible market research reports and the potential benefits of AI and ML in education.


Example 1 showed us the raw multi-agent communication agents sending, receiving, and logging messages step-by-step, all manually controlled.



Now Example 2 takes that same logic but wraps it into a LangGraph-powered workflow, where the routing and coordination happen automatically.

In [16]:
from typing import TypedDict, Literal, Dict, List
from langgraph.graph import StateGraph, END
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from dataclasses import dataclass

# --------------------------------------------------------
#  DEFINE AGENT IDENTITIES AND MESSAGE STRUCTURE
# --------------------------------------------------------

# Allowed agent names (restricts to 4 roles only)
AgentID = Literal["Supervisor", "Researcher", "Analyst", "Writer"]

# Define message structure between agents (FIPA-style)
@dataclass
class Msg:
    sender: AgentID                       # Who sent the message
    receiver: AgentID                     # Who will receive it
    performative: Literal["request","inform","propose","confirm","reject"]  # Type of message
    content: str                          # Actual text / information
    conv_id: str                          # Conversation ID (to link related messages)

# --------------------------------------------------------
#  DEFINE THE STATE SHARED BY THE SYSTEM (LangGraph)
# --------------------------------------------------------

class S(TypedDict, total=False):
    mailboxes: Dict[AgentID, List[Msg]]   # Each agent's inbox
    log: List[Msg]                        # Permanent record of all exchanged messages
    question: str                         # The main query or topic of discussion

# --------------------------------------------------------
#  SET UP SEPARATE LLMs (PRIVATE BRAINS PER AGENT)
# --------------------------------------------------------

# Each agent gets its own ChatGroq instance (different temperature = different personality)
LLM = {
    "Supervisor": ChatGroq(model="llama-3.1-8b-instant", temperature=0),   # Logical coordinator
    "Researcher": ChatGroq(model="llama-3.1-8b-instant", temperature=0.2), # Exploratory info seeker
    "Analyst":    ChatGroq(model="llama-3.1-8b-instant", temperature=0),   # Strict verifier
    "Writer":     ChatGroq(model="llama-3.1-8b-instant", temperature=0.3), # Creative summarizer
}

# System prompts define the policy / role of each agent
SYS = {
    "Researcher": "You are a rigorous market researcher. 3–5 numbered findings with short sources.",
    "Analyst":    "You are a strict analyst. Remove weak/uncited claims; justify each remaining.",
    "Writer":     "You are a concise report writer. ~120-word executive summary, neutral tone.",
}

# --------------------------------------------------------
# DEFINE BASIC MESSAGE SEND/RECEIVE BEHAVIOR
# --------------------------------------------------------

# Utility: Call an agent's LLM with its system prompt + new input
def call(agent: AgentID, sys: str, user: str) -> str:
    """Send user message to specific agent and return the model’s text output."""
    return LLM[agent].invoke([SystemMessage(content=sys),
                              HumanMessage(content=user)]).content

# Utility: Pop (read & remove) first message from an agent's mailbox
def pop_mail(s: S, who: AgentID) -> Msg | None:
    """Retrieve the oldest unread message for a given agent."""
    box = s["mailboxes"].setdefault(who, [])
    return box.pop(0) if box else None

# Utility: Push (send) a new message into receiver's mailbox
def push_mail(s: S, m: Msg):
    """Deliver a message to the target agent’s inbox."""
    s["mailboxes"].setdefault(m.receiver, []).append(m)

# --------------------------------------------------------
#  DEFINE NODES (AGENT BEHAVIOR FUNCTIONS)
# --------------------------------------------------------

def supervisor_node(s: S) -> S:
    """Supervisor coordinates workflow: decides which agent to contact next."""
    log = s.get("log", [])
    has_r = any(m.sender == "Researcher" for m in log)   # Has researcher responded?
    has_a = any(m.sender == "Analyst" for m in log)      # Has analyst responded?

    # Step 1 → Ask Researcher if no research done yet
    if not has_r:
        push_mail(s, Msg("Supervisor", "Researcher", "request", s["question"], "MR-2001"))

    # Step 2 → Ask Analyst once research findings exist
    elif not has_a:
        last_findings = next(m for m in reversed(log) if m.sender == "Researcher").content
        push_mail(s, Msg("Supervisor", "Analyst", "request", last_findings, "MR-2001"))

    # Step 3 → Ask Writer after Analyst verification
    else:
        last_verified = next(m for m in reversed(log) if m.sender == "Analyst").content
        push_mail(s, Msg("Supervisor", "Writer", "request", last_verified, "MR-2001"))
    return s

def researcher_node(s: S) -> S:
    """Researcher performs market research and sends findings to Supervisor."""
    m = pop_mail(s, "Researcher")
    if not m: return s  # No message? Skip turn
    out = call("Researcher", SYS["Researcher"], f"QUESTION: {m.content}")
    msg = Msg("Researcher", "Supervisor", "inform", out, m.conv_id)
    s.setdefault("log", []).append(msg)   # Save to global log
    push_mail(s, msg)                     # Send back to Supervisor
    return s

def analyst_node(s: S) -> S:
    """Analyst verifies researcher’s findings for accuracy and strength."""
    m = pop_mail(s, "Analyst")
    if not m: return s
    out = call("Analyst", SYS["Analyst"], f"VERIFY & CLEAN:\n{m.content}")
    msg = Msg("Analyst", "Supervisor", "inform", out, m.conv_id)
    s.setdefault("log", []).append(msg)
    push_mail(s, msg)
    return s

def writer_node(s: S) -> S:
    """Writer creates final summary report from verified notes."""
    m = pop_mail(s, "Writer")
    if not m: return s
    out = call("Writer", SYS["Writer"], f"SOURCE NOTES:\n{m.content}")
    msg = Msg("Writer", "Supervisor", "inform", out, m.conv_id)
    s.setdefault("log", []).append(msg)
    push_mail(s, msg)
    return s

# --------------------------------------------------------
#  ROUTER FUNCTION – DECIDES NEXT NODE TO RUN
# --------------------------------------------------------

def router(s: S) -> Literal["supervisor","researcher","analyst","writer","end"]:
    """Router inspects mailboxes and decides which agent acts next."""
    # Priority: if someone has mail waiting, let them act
    for who in ["Researcher", "Analyst", "Writer"]:
        if s["mailboxes"].get(who):
            return who.lower()

    # If no pending mail, let Supervisor plan the next step
    # End condition: stop when Writer has already informed Supervisor
    if any(m.sender == "Writer" for m in s.get("log", [])):
        return "end"
    return "supervisor"

# --------------------------------------------------------
#  BUILD THE LANGGRAPH WORKFLOW
# --------------------------------------------------------

g = StateGraph(S)                              # Create a new state graph using our TypedDict
g.add_node("supervisor", supervisor_node)      # Add each agent as a node
g.add_node("researcher", researcher_node)
g.add_node("analyst", analyst_node)
g.add_node("writer", writer_node)

g.set_entry_point("supervisor")                # Starting point: Supervisor decides first step

# Conditional edges: router decides who runs next after each node
for n in ["supervisor", "researcher", "analyst", "writer"]:
    g.add_conditional_edges(n, router, {
        "supervisor": "supervisor",
        "researcher": "researcher",
        "analyst": "analyst",
        "writer": "writer",
        "end": END
    })

# Compile graph into an executable MAS app
app = g.compile()

# --------------------------------------------------------
#  RUN THE SYSTEM
# --------------------------------------------------------

# Initialize starting state (question + empty mailboxes + empty log)
state = app.invoke({
    "question": "Pakistani K–12 EdTech market (2024–2025): size, growth, top 3 trends. Cite concise sources.",
    "mailboxes": {},
    "log": []
})

# Find Writer’s final message in the log and print summary
final = next((m for m in state["log"][::-1] if m.sender == "Writer"), None)
print("[FINAL SUMMARY]\n", final.content if final else "(none)")


[FINAL SUMMARY]
 **Executive Summary (120 words)**

The Pakistani K-12 EdTech market is expected to reach PKR 34.6 billion (approximately USD 170 million) by 2025, with a CAGR of 23.1% from 2020 to 2025. This growth is driven by increasing internet penetration, government initiatives, and demand for online learning platforms. Key trends in the market include the integration of Artificial Intelligence (AI) to enable personalized learning experiences, the use of gamification and interactive content to make learning more engaging, and the growth of mobile learning. These findings are based on credible market research reports from ResearchAndMarkets.com and Grand View Research, which provide a comprehensive analysis of the Pakistani EdTech market.
